# Decision Trees: Building, Understanding, and Evaluating Classifiers

**Upper-level Artificial Intelligence / Machine Learning Lab**

**Estimated time:** 90–120 minutes  
**Tools:** Python, pandas, matplotlib, scikit-learn, Jupyter

## Lab overview

In this lab, you will build a decision-tree classifier using the Wisconsin Breast Cancer dataset. You will investigate how trees choose splits, compare Gini impurity with entropy, visualize a trained tree, observe overfitting, experiment with pruning, and interpret model performance.

The dataset contains measurements computed from digitized images of breast mass samples. The task is to classify each sample as **malignant** or **benign**. This is an educational exercise, not a clinical diagnostic system.

## Learning objectives

By the end of the lab, you should be able to:

1. Explain how a decision tree makes predictions.
2. Calculate and interpret Gini impurity and entropy.
3. Explain how a tree evaluates candidate splits.
4. Train and visualize a decision tree with scikit-learn.
5. Identify overfitting by comparing training and test performance.
6. Experiment with depth limits and cost-complexity pruning.
7. Interpret feature importance and a confusion matrix.
8. Discuss the strengths and limitations of decision trees.

## How to complete the lab

Run the notebook cells in order. Some cells ask you to change a parameter, compare results, or write a short explanation. Record your answers in the response areas. Do not simply report numbers: explain what they mean.

**Suggested submission:** This completed notebook, including code, plots, and written responses.


## Activity 0 — Setup and imports

We will use scikit-learn's built-in dataset, so no separate data download is required.

Run the following cell. If an import fails, install the missing package in your Python environment (for example, `pip install scikit-learn pandas matplotlib`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

print("Setup complete.")


## Activity 1 — Explore the dataset

The dataset has 30 numeric features describing characteristics of cell nuclei. The target contains two classes. In scikit-learn's encoding, the target names are `malignant` and `benign`; the numeric labels correspond to the order shown by `data.target_names`.

First, load and inspect the data.


In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Number of observations:", X.shape[0])
print("Number of features:", X.shape[1])
print("Target names:", data.target_names)

display(X.head())


In [ ]:
print("Class counts:")
print(y.value_counts().sort_index().rename(index=dict(enumerate(data.target_names))))

print("\nFeature summary:")
display(X.describe().T)


### Questions — Activity 1

1. How many observations and features are in the dataset?
2. How many examples belong to each class? Is the dataset perfectly balanced?
3. Pick three features that you think might help distinguish malignant from benign samples. Explain your choices.
4. Why should we keep a separate test set rather than evaluate a model only on the data used to train it?

**Your answers:**

_Write your responses in this cell._


## Activity 2 — How decision trees measure a good split

A decision tree makes a prediction by asking a sequence of questions about input features. Each question divides the data into child groups, called **nodes**. The tree prefers splits that make the resulting groups more homogeneous—that is, groups in which observations mostly belong to one class.

### 2.1 What is impurity?

**Impurity** describes how mixed the class labels are within a node.

Imagine a node containing 10 tumor samples:

- 10 benign and 0 malignant: the node is completely pure.
- 8 benign and 2 malignant: the node is mostly one class, but still mixed.
- 5 benign and 5 malignant: the node is maximally mixed for a two-class problem.

A pure node has impurity **0**. A node with a balanced mix of classes has higher impurity.

### 2.2 Gini impurity

Gini impurity estimates the chance of incorrectly labeling an observation if we randomly assign a class label according to the class proportions in that node.

\[
Gini = 1 - \sum_{i=1}^{K} p_i^2
\]

Here, \(p_i\) is the proportion of observations belonging to class \(i\), and \(K\) is the number of classes.

For a node with 8 benign and 2 malignant samples:

\[
Gini = 1 - (0.8^2 + 0.2^2) = 1 - (0.64 + 0.04) = 0.32
\]

For a node with 5 benign and 5 malignant samples:

\[
Gini = 1 - (0.5^2 + 0.5^2) = 0.50
\]

So the balanced node is more impure.

### 2.3 Entropy

**Entropy** measures uncertainty or disorder in the class labels. It is 0 when the node contains only one class. It is highest when the classes are evenly distributed.

\[
H = -\sum_{i=1}^{K} p_i \log_2(p_i)
\]

For 8 benign and 2 malignant samples:

\[
H = -(0.8\log_2 0.8 + 0.2\log_2 0.2) \approx 0.722
\]

For a 50/50 split:

\[
H = -(0.5\log_2 0.5 + 0.5\log_2 0.5) = 1.0
\]

If a class proportion is zero, its contribution is treated as zero (because the limit of \(p\log p\) as \(p\) approaches zero is zero).

### 2.4 Compare the measures

For a two-class problem, the following examples illustrate how both measures increase as the classes become more evenly mixed:

| Class distribution | Gini impurity | Entropy (bits) |
|---|---:|---:|
| 10 benign, 0 malignant | 0.00 | 0.00 |
| 8 benign, 2 malignant | 0.32 | 0.72 |
| 5 benign, 5 malignant | 0.50 | 1.00 |

The numerical scales differ, but both measures express a similar idea: **lower values mean a purer node**.

### 2.5 How does a tree use impurity?

A candidate split divides a parent node into child nodes. The tree calculates the **weighted average impurity** of those children:

\[
I_{after} =
\frac{N_L}{N} I(L) +
\frac{N_R}{N} I(R)
\]

where \(N_L\) and \(N_R\) are the numbers of samples in the left and right child nodes, and \(N\) is the number in the parent.

The reduction in impurity is:

\[
\text{Impurity reduction} = I(parent) - I_{after}
\]

For entropy, this reduction is called **information gain**. The tree prefers a split that produces a larger reduction in impurity. In practice, it searches over features and possible threshold values to find a strong split.

### 2.6 Calculate impurity yourself

The next cell implements both measures. Run it, then verify the values against the examples above.


In [ ]:
def gini_impurity(class_counts):
    """Calculate Gini impurity from a sequence of class counts."""
    counts = np.asarray(class_counts, dtype=float)
    probabilities = counts / counts.sum()
    return 1 - np.sum(probabilities ** 2)


def entropy(class_counts):
    """Calculate entropy in bits from a sequence of class counts."""
    counts = np.asarray(class_counts, dtype=float)
    probabilities = counts / counts.sum()
    probabilities = probabilities[probabilities > 0]
    return -np.sum(probabilities * np.log2(probabilities))


examples = {
    "10 benign, 0 malignant": [10, 0],
    "8 benign, 2 malignant": [8, 2],
    "5 benign, 5 malignant": [5, 5],
}

for description, counts in examples.items():
    print(
        f"{description:27s} "
        f"Gini = {gini_impurity(counts):.3f}, "
        f"Entropy = {entropy(counts):.3f}"
    )


### 2.7 Compare two possible splits

Suppose a parent node has 10 observations: 5 benign and 5 malignant. Consider two candidate splits:

- **Split A:** child 1 has 5 benign; child 2 has 5 malignant.
- **Split B:** child 1 has 4 benign and 1 malignant; child 2 has 1 benign and 4 malignant.

Calculate the weighted child impurity for each split using both Gini and entropy. Which split is better?


In [ ]:
parent_counts = [5, 5]

# Each split is represented as (left-child counts, right-child counts).
split_A = ([5, 0], [0, 5])
split_B = ([4, 1], [1, 4])

def weighted_child_impurity(split, impurity_function):
    left, right = split
    total = sum(left) + sum(right)
    return (
        sum(left) / total * impurity_function(left)
        + sum(right) / total * impurity_function(right)
    )

for name, split in [("A", split_A), ("B", split_B)]:
    gini_after = weighted_child_impurity(split, gini_impurity)
    entropy_after = weighted_child_impurity(split, entropy)
    print(
        f"Split {name}: weighted Gini = {gini_after:.3f}; "
        f"weighted entropy = {entropy_after:.3f}"
    )

print("Parent Gini:", gini_impurity(parent_counts))
print("Parent entropy:", entropy(parent_counts))


### Questions — Activity 2

1. In your own words, what does impurity tell us about a node?
2. What is the Gini impurity of a node with 8 benign and 2 malignant samples?
3. What is the entropy of a node with 8 benign and 2 malignant samples?
4. Which is more impure: a node containing 8/2 samples or one containing 5/5 samples? Explain.
5. Which candidate split (A or B) is better? Explain using the weighted child impurity.
6. What is information gain?
7. Why might a decision tree prefer a split that creates one small, very pure child and one large, somewhat mixed child—or reject it? Consider the weighted average.

**Your answers:**

_Write your responses in this cell._


## Activity 3 — Train and visualize your first decision tree

We will split the data into training and test sets. The `stratify=y` argument preserves approximately the same class proportions in both sets. The test set is held back until evaluation.

Then we will train a decision tree using Gini impurity.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).sort_index())


In [ ]:
tree_gini = DecisionTreeClassifier(
    criterion="gini",
    random_state=RANDOM_STATE
)

tree_gini.fit(X_train, y_train)

train_predictions = tree_gini.predict(X_train)
test_predictions = tree_gini.predict(X_test)

print("Training accuracy:", accuracy_score(y_train, train_predictions))
print("Test accuracy:", accuracy_score(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=data.target_names
))


### Visualize the tree

A decision-tree visualization shows the feature and threshold used at each internal node. A path from the root to a leaf represents a sequence of decisions. The leaf gives the predicted class.

In the visualization:
- `gini` is the node's Gini impurity.
- `samples` is the number of training observations reaching the node.
- `value` shows the class counts at the node.
- `class` is the predicted class.


In [ ]:
plt.figure(figsize=(24, 12))
plot_tree(
    tree_gini,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree — Gini Impurity")
plt.show()

print("Tree depth:", tree_gini.get_depth())
print("Number of leaves:", tree_gini.get_n_leaves())


### Questions — Activity 3

1. What feature is used at the root (top) of the tree?
2. What threshold does the root node use?
3. How many levels deep is the tree? How many leaf nodes does it have?
4. Pick one test observation. Trace the feature comparisons from the root to a leaf. What class does the tree predict?
5. Does a high training accuracy guarantee that the model will perform well on new data? Why or why not?

**Your answers:**

_Write your responses in this cell._


In [ ]:
# Optional helper: inspect one test observation and its prediction.
sample_position = 0
sample = X_test.iloc[[sample_position]]

print("Actual class:", data.target_names[int(y_test.iloc[sample_position])])
print("Predicted class:", data.target_names[int(tree_gini.predict(sample)[0])])
display(sample)


## Activity 4 — Compare Gini impurity and entropy

Scikit-learn supports both criteria. Train one tree with each criterion using the same training data and random state. Compare their performance and structure.

The two criteria often produce similar trees, but they are not guaranteed to select identical splits.


In [ ]:
models_by_criterion = {}

for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(
        criterion=criterion,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    models_by_criterion[criterion] = model

    print(f"Criterion: {criterion}")
    print("  Training accuracy:", model.score(X_train, y_train))
    print("  Test accuracy:", model.score(X_test, y_test))
    print("  Depth:", model.get_depth())
    print("  Leaves:", model.get_n_leaves())


### Questions — Activity 4

1. Did Gini and entropy produce trees with the same depth and number of leaves?
2. Did one criterion achieve higher test accuracy?
3. Does a small difference in accuracy mean one criterion is always better? Why is it important to evaluate on more than one split?

**Your answers:**

_Write your responses in this cell._


## Activity 5 — Investigate overfitting

A decision tree can keep splitting until it creates very specific rules for the training observations. A deep tree may achieve nearly perfect training accuracy while performing less well on unseen data. This is **overfitting**.

The `max_depth` parameter limits the number of levels in the tree. We will train trees with different maximum depths and compare training and test accuracy.


In [ ]:
depths = range(1, 21)
train_scores = []
test_scores = []

for depth in depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)

    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, marker="o", label="Training accuracy")
plt.plot(depths, test_scores, marker="o", label="Test accuracy")
plt.xlabel("Maximum tree depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Training vs. Test Accuracy")
plt.xticks(list(depths))
plt.ylim(0.85, 1.01)
plt.grid(True)
plt.legend()
plt.show()


### Questions — Activity 5

1. At what depth does training accuracy first reach 100%, if it does?
2. Does test accuracy keep increasing as depth increases?
3. Identify a depth where the gap between training and test accuracy suggests overfitting.
4. Why can a tree with perfect training accuracy perform worse on new data?
5. Based on this experiment, would you choose the deepest tree? Explain.

**Your answers:**

_Write your responses in this cell._


## Activity 6 — Control complexity with pruning

Limiting maximum depth is one way to control a tree's complexity. Another is **cost-complexity pruning**.

The pruning objective can be written as:

\[
R_\alpha(T) = R(T) + \alpha |T|
\]

Here, \(R(T)\) represents the tree's total leaf error (as defined by the pruning procedure), \(|T|\) is the number of terminal leaves, and \(\alpha\) controls the penalty for tree complexity. Larger values of `ccp_alpha` generally produce smaller trees.

We will compare several values. The best value depends on the data and should be selected using validation or cross-validation rather than by repeatedly optimizing against the final test set.


In [ ]:
alphas = [0.0, 0.0001, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]

pruning_results = []

for alpha in alphas:
    model = DecisionTreeClassifier(
        ccp_alpha=alpha,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)

    pruning_results.append({
        "ccp_alpha": alpha,
        "depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "training_accuracy": model.score(X_train, y_train),
        "test_accuracy": model.score(X_test, y_test)
    })

pruning_df = pd.DataFrame(pruning_results)
display(pruning_df)


### Cross-validation for a more reliable comparison

A single train/test split can give a noisy estimate of model performance. Cross-validation repeatedly trains and evaluates models on different partitions of the training data. Here, we use five-fold cross-validation on the training set to compare pruning values.

The held-out test set remains separate for the final evaluation.


In [ ]:
cv_results = []

for alpha in alphas:
    model = DecisionTreeClassifier(
        ccp_alpha=alpha,
        random_state=RANDOM_STATE
    )
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    cv_results.append({
        "ccp_alpha": alpha,
        "mean_cv_accuracy": scores.mean(),
        "std_cv_accuracy": scores.std()
    })

cv_df = pd.DataFrame(cv_results)
display(cv_df.sort_values("mean_cv_accuracy", ascending=False))


### Select and evaluate a pruned model

Choose the `ccp_alpha` with the highest mean cross-validation accuracy. If several values are very close, consider preferring the simpler tree. Fit that model on the full training set and evaluate it once on the held-out test set.


In [ ]:
best_alpha = cv_df.loc[cv_df["mean_cv_accuracy"].idxmax(), "ccp_alpha"]

pruned_tree = DecisionTreeClassifier(
    ccp_alpha=float(best_alpha),
    random_state=RANDOM_STATE
)
pruned_tree.fit(X_train, y_train)

print("Selected ccp_alpha:", best_alpha)
print("Tree depth:", pruned_tree.get_depth())
print("Number of leaves:", pruned_tree.get_n_leaves())
print("Training accuracy:", pruned_tree.score(X_train, y_train))
print("Held-out test accuracy:", pruned_tree.score(X_test, y_test))


In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    pruned_tree,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title(f"Pruned Decision Tree (ccp_alpha={best_alpha:.5f})")
plt.show()


### Questions — Activity 6

1. As `ccp_alpha` increases, what generally happens to tree depth and number of leaves?
2. Does pruning always improve test accuracy?
3. Why use cross-validation to choose a pruning value?
4. Which model would you choose: the most accurate model, the simplest model, or a compromise? Defend your choice using your results.
5. Why should the final test set not be used repeatedly to choose hyperparameters?

**Your answers:**

_Write your responses in this cell._


## Activity 7 — Feature importance and model interpretation

Decision trees can report feature importance scores. In scikit-learn, these scores are based on the total (weighted) reduction in the splitting criterion attributable to each feature across the tree, then normalized to sum to 1.

Feature importance can help summarize which features the tree used. However, it does **not** establish that a feature causes the outcome. Importance can also be distributed among correlated features, and impurity-based importance can favor some kinds of features.

We will inspect the most important features in the pruned tree.


In [ ]:
importance = pd.Series(
    pruned_tree.feature_importances_,
    index=data.feature_names
).sort_values(ascending=False)

display(importance.head(10).to_frame("importance"))

top_importance = importance.head(10).sort_values()
plt.figure(figsize=(10, 6))
top_importance.plot(kind="barh")
plt.xlabel("Feature importance")
plt.title("Top 10 Features — Pruned Decision Tree")
plt.show()


### Questions — Activity 7

1. Which three features have the highest importance in your model?
2. Are these the same features you predicted would matter in Activity 1?
3. What does a feature importance score tell you?
4. Why should you not interpret feature importance as proof of causation?
5. What might happen to importance scores when two features are strongly correlated?

**Your answers:**

_Write your responses in this cell._


## Activity 8 — Evaluate errors, not just accuracy

Accuracy is the fraction of predictions that are correct. It is useful, but it does not tell us which kinds of mistakes the model makes.

A **confusion matrix** compares actual classes with predicted classes. A false negative in this lab means a malignant sample was predicted to be benign. A false positive means a benign sample was predicted to be malignant.

For a medical screening context, a false negative can be especially concerning. This is why recall for the malignant class may matter more than accuracy alone. The appropriate tradeoff depends on the real-world setting; this classroom model is not suitable for clinical use.


In [ ]:
final_predictions = pruned_tree.predict(X_test)

print(classification_report(
    y_test,
    final_predictions,
    target_names=data.target_names
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_predictions,
    display_labels=data.target_names,
    cmap="Blues"
)
plt.title("Confusion Matrix — Pruned Decision Tree")
plt.show()


### Questions — Activity 8

1. What is the model's accuracy on the held-out test set?
2. What is precision? What is recall?
3. In this dataset, which class corresponds to a malignant sample?
4. How many malignant samples were incorrectly classified as benign?
5. Why might recall for the malignant class be particularly important in a screening context?
6. Could a model have high accuracy but still be unsuitable for this task? Explain.

**Your answers:**

_Write your responses in this cell._


## Activity 9 — Advanced challenges

Complete one or more of these challenges if time permits, or use them as follow-up assignments.

### Challenge A: Change the train/test split

Change `test_size` or `random_state` in Activity 3. Retrain the model. How much do its accuracy, root feature, depth, and number of leaves change?

### Challenge B: Restrict the features

Train a tree using only five features of your choice. Compare its performance and complexity with the tree that uses all 30 features. Explain how you selected the five.

### Challenge C: Change other complexity parameters

Experiment with `min_samples_split` and `min_samples_leaf`. How do they affect tree size and generalization?

Example:

```python
DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)
```

### Challenge D: Compare a single tree with a random forest

A random forest trains many decision trees on different samples and combines their predictions. Compare a single decision tree with a random forest. What happens to predictive performance and interpretability?

### Challenge E: Investigate model stability

Train trees using several random seeds or different training subsets. Compare their root features and structures. If the trees differ, what does that suggest about the stability of a decision tree explanation?

### Challenge F: Gini versus entropy beyond accuracy

Compare the structures of Gini and entropy trees. Do they make the same predictions? Are their most important features the same? Explain why similar performance does not necessarily imply identical models.


## Final reflection

Write a short conclusion (approximately 200–300 words) addressing the following:

1. How does a decision tree choose a split?
2. What is the difference between Gini impurity and entropy?
3. What evidence of overfitting did you observe?
4. How did pruning affect model complexity and performance?
5. What are the main advantages and limitations of decision trees?
6. In what situations might you prefer a decision tree over a more complex model?

## Submission checklist

- [ ] All notebook cells have been run.
- [ ] Activities 1–8 include written answers.
- [ ] The notebook includes the requested visualizations.
- [ ] The final reflection is complete.
- [ ] I can explain impurity, entropy, information gain, and overfitting in my own words.
